# Post-hoc analysis — HITL Text-to-SQL

**This notebook is the analysis half only.** The live participant session runs in the web
app (`backend/` + `frontend/`); here we load the per-session logs it wrote, run the C1
baseline, score against hand-written gold SQL, and produce the statistics.

Scoring and thematic analysis are **not** automated — the cells scaffold them for you.

## Cell 1 — Load logs into pydantic objects

In [ ]:
import sys, pathlib, json
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'analysis' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))

from backend import config
from backend.logging_io.session_log import SessionLog

log_dir = config.session_log_dir()
sessions = []
for p in sorted(log_dir.glob('p*.json')):
    try:
        sessions.append(SessionLog.model_validate_json(p.read_text()))
    except Exception as e:
        print('skip', p.name, e)
print(f'Loaded {len(sessions)} session(s) from {log_dir}')

## Cell 2 — Flatten to a long `trials_df`
One row per trial, with empty columns for the hand-scored fields.

In [ ]:
import pandas as pd

rows = []
for s in sessions:
    for t in s.trials:
        perceived = t.perceived_success or {}
        result = t.result or {}
        gen_calls = [c for c in t.calls if c.call == '01_interface_generation']
        rows.append({
            'participant_id': s.participant_id,
            'trial_id': t.trial_id,
            'db_name': t.db_name,
            'ambiguity_class': t.ambiguity_class,
            'condition': t.condition,
            'question': t.question,
            'intent_note': t.intent_note,
            'sql': result.get('sql'),
            'exec_success': result.get('success'),
            'row_count': result.get('row_count'),
            'compile_success': t.compile_success,
            'interface_confidence': perceived.get('interface_confidence'),
            'answer_wanted': perceived.get('answer_wanted'),
            'answer_confidence': perceived.get('answer_confidence'),
            'interface_tokens': sum(c.input_tokens + c.output_tokens for c in gen_calls),
            'interface_latency_ms': sum(c.latency_ms for c in gen_calls),
            # hand-scored later:
            'actual_success': None,        # {correct, partial, incorrect}
            'failure_attribution': None,
            'false_confidence_flag': None,
        })
trials_df = pd.DataFrame(rows)
print(trials_df.shape)
trials_df.head()

## Cell 3 — C1 baseline (no human)
Run `c1_baseline.py` against each authored question and append `condition='C1'` rows.
Stubbed model, real execution — flip `USE_STUB=False` in `config.py` to go live.

In [ ]:
from backend.conditions.c1_baseline import run_baseline

c1_rows = []
seen = set()
for s in sessions:
    for t in s.trials:
        key = (t.db_name, t.question)
        if key in seen or not t.question:
            continue
        seen.add(key)
        out = run_baseline(t.db_name, t.question)
        c1_rows.append({
            'participant_id': s.participant_id, 'trial_id': f'{t.trial_id}_C1',
            'db_name': t.db_name, 'ambiguity_class': t.ambiguity_class, 'condition': 'C1',
            'question': t.question, 'intent_note': t.intent_note,
            'sql': out['sql'], 'exec_success': out['execution']['success'],
            'row_count': out['execution']['row_count'],
            'actual_success': None, 'failure_attribution': None, 'false_confidence_flag': None,
        })
c1_df = pd.DataFrame(c1_rows)
all_df = pd.concat([trials_df, c1_df], ignore_index=True) if len(c1_df) else trials_df.copy()
print(f'Appended {len(c1_df)} C1 baseline rows; {len(all_df)} rows total')

## Cell 4 — Gold-SQL scaffold + scoring helpers
Writes one `gold_sql/p{NN}_t{NN}.md` stub per trial (git-ignored). You fill in the gold
SQL and the rating; `score_trial` records `actual_success`, the false-confidence flag and
failure attribution. `cohens_kappa` reads a second rater's CSV. **Nothing here auto-scores.**

In [ ]:
GOLD = ROOT / 'gold_sql'
GOLD.mkdir(exist_ok=True)

def gold_path(row):
    return GOLD / f"p{int(row['participant_id']):02d}_{row['trial_id']}.md"

def write_gold_scaffold(df):
    n = 0
    for _, row in df.iterrows():
        path = gold_path(row)
        if path.exists():
            continue
        path.write_text(
            f"# Gold SQL — {row['trial_id']} ({row['condition']}, {row['ambiguity_class']})\n\n"
            f"## Question\n{row['question']}\n\n## Intent note (scoring anchor)\n{row['intent_note']}\n\n"
            f"## System SQL\n```sql\n{row['sql']}\n```\n\n"
            "## Gold SQL (fill in)\n```sql\n-- TODO\n```\n\n"
            "## Rating: correct | partial | incorrect\nTODO\n\n## Failure attribution (if any)\nTODO\n"
        )
        n += 1
    return n

print('wrote', write_gold_scaffold(all_df), 'gold scaffold file(s)')

def score_trial(df, trial_id, actual_success, failure_attribution=None):
    """Record a hand rating. false_confidence_flag = confident but not correct."""
    mask = df['trial_id'] == trial_id
    df.loc[mask, 'actual_success'] = actual_success
    df.loc[mask, 'failure_attribution'] = failure_attribution
    confident = df.loc[mask, 'answer_confidence'].isin(['Very', 'Completely'])
    df.loc[mask, 'false_confidence_flag'] = bool(confident.any()) and actual_success != 'correct'
    return df

def cohens_kappa(csv_path=GOLD / 'second_rater.csv'):
    import pandas as pd
    if not pathlib.Path(csv_path).exists():
        print('no second_rater.csv yet'); return None
    sr = pd.read_csv(csv_path)
    merged = all_df.merge(sr, on='trial_id', suffixes=('', '_r2')).dropna(subset=['actual_success', 'actual_success_r2'])
    if merged.empty:
        print('no overlapping scored rows'); return None
    cats = sorted(set(merged['actual_success']) | set(merged['actual_success_r2']))
    po = (merged['actual_success'] == merged['actual_success_r2']).mean()
    pe = sum((merged['actual_success'] == c).mean() * (merged['actual_success_r2'] == c).mean() for c in cats)
    kappa = (po - pe) / (1 - pe) if pe != 1 else float('nan')
    print(f'Cohen\u2019s kappa = {kappa:.3f} (n={len(merged)})')
    return kappa

## Cell 5 — Descriptives
Success by condition + class, the perceived-vs-actual gap, the 2×2 reliance matrix per
condition, and the failure-attribution distribution. Runs on whatever has been scored.

In [ ]:
def success_by(df, by):
    scored = df.dropna(subset=['actual_success'])
    if scored.empty:
        print('No trials scored yet — fill in gold_sql/*.md and call score_trial().'); return None
    g = scored.assign(correct=scored['actual_success'].eq('correct'))
    return g.groupby(by)['correct'].agg(['mean', 'count'])

print('Execution success by condition (proxy until scored):')
display(all_df.groupby('condition')['exec_success'].agg(['mean', 'count']))
print('\nActual success by condition (once scored):')
print(success_by(all_df, 'condition'))
print('\nActual success by condition + class (once scored):')
print(success_by(all_df, ['condition', 'ambiguity_class']))

def reliance_matrix(df, condition):
    """2x2: perceived (answer_wanted == 'Yes') x actual (correct) -> reliance quadrant."""
    d = df[(df['condition'] == condition)].dropna(subset=['actual_success', 'answer_wanted'])
    if d.empty:
        return f'{condition}: nothing scored yet'
    perceived_ok = d['answer_wanted'].eq('Yes')
    actual_ok = d['actual_success'].eq('correct')
    import pandas as pd
    return pd.crosstab(perceived_ok.map({True: 'felt right', False: 'felt wrong'}),
                       actual_ok.map({True: 'was correct', False: 'was wrong'}))

for c in ['C2', 'C3']:
    print(f'\nReliance matrix — {c}:'); print(reliance_matrix(all_df, c))

## Cells 6–10 — Calibration, SUS + agency, cost, secondary model, qual export
Scaffolded below. The SUS and cost cells run on stub data already; calibration and the
secondary model need scored `actual_success`.

In [ ]:
# --- Cell 7: SUS + agency (per condition) ---
def sus_score(items):
    """Brooke 1996: odd items (r-1), even items (5-r), summed x2.5 -> 0..100."""
    if len(items) < 10:
        return None
    total = 0
    for i in range(1, 11):
        r = items.get(str(i), items.get(i))
        if r is None:
            return None
        total += (r - 1) if i % 2 == 1 else (5 - r)
    return total * 2.5

q_rows = []
for s in sessions:
    for q in s.questionnaires:
        q_rows.append({'participant_id': s.participant_id, 'condition': q.get('condition'),
                       'sus': sus_score(q.get('sus', {}))})
sus_df = pd.DataFrame(q_rows)
print('Mean SUS by condition:')
print(sus_df.groupby('condition')['sus'].agg(['mean', 'std', 'count']) if not sus_df.empty else '(no questionnaires)')

In [ ]:
# --- Cell 8: Cost (tokens + latency) and C3 compile-success rate ---
print('Interface-generation cost by condition:')
display(all_df.groupby('condition')[['interface_tokens', 'interface_latency_ms']].agg(['mean', 'std']))
c3 = trials_df[trials_df['condition'] == 'C3']
if len(c3):
    print('C3 compile-success rate:', c3['compile_success'].mean())

In [ ]:
# --- Cell 10: Qualitative export (open-text + per-trial question vs output) ---
qual = all_df[['participant_id', 'trial_id', 'condition', 'ambiguity_class', 'question',
               'intent_note', 'sql', 'answer_wanted', 'answer_confidence']].copy()
out = ROOT / 'analysis' / 'qual_export.csv'
qual.to_csv(out, index=False)
print('wrote', out)

### Counterbalance table (reproduced from `backend/counterbalance/assign.py`)

In [ ]:
from backend.counterbalance.assign import all_assignments, describe
for a in all_assignments():
    print(describe(a)); print()